## Final Dataset

In [1]:
# Load data
import pandas as pd
data = pd.read_csv('sa_in_boundary.csv')

# date/time to proper type
data['datetime'] = pd.to_datetime(data['date'] + ' ' + data['time'])

# reorder
cols = ['datetime'] + [c for c in data.columns if c != 'datetime']
data = data[cols]

# drop unwanted columns
data = data.drop(columns=['date', 'time', 'geocode_source', 'geometry'])

In [2]:
# View
data.head()

,datetime,lat,lng,district,subject_age,violation,arrest_made,citation_issued,search_conducted,speed,subject_race_black,subject_race_hispanic,subject_race_other,subject_race_white,subject_sex_male,extreme_cases
0,2014-01-01 23:35:00,29.419450,-98.456763,4140.0,38.0,CROSSING AGAINST PEDESTRIAN CONTROL SIGNAL|PED...,0,1,0,NaN,0,1,0,0,1,0
1,2014-01-01 22:48:00,29.511846,-98.538947,7210.0,47.0,DISREGARD STOP SIGN|OPERATING A MOTOR VEHICLE ...,0,1,0,NaN,0,1,0,0,0,0
2,2014-01-01 08:30:00,29.549645,-98.463889,3130.0,27.0,OPERATING A MOTOR VEHICLE WITHOUT A VALID LICENSE,0,1,0,NaN,0,1,0,0,1,0
3,2014-01-01 07:51:00,29.352803,-98.529048,6150.0,27.0,SPEEDING-POSTED LIMIT|DRIVING WITHOUT PROOF OF...,0,1,0,78.0,0,1,0,0,1,0
4,2014-01-01 12:27:00,29.424226,-98.442586,4160.0,28.0,OPERATING A MOTOR VEHICLE WITHOUT A VALID LICE...,0,1,0,NaN,1,0,0,0,1,0


## Combine with SOPP policy data

In [3]:
# combine with policy data
policy_data = pd.read_csv('policy_data.csv')
policy_data = policy_data.iloc[:, :6]

# make start/end dates datetime
policy_data['effective_date'] = pd.to_datetime(policy_data['effective_date'], errors='coerce')
policy_data['end_date'] = pd.to_datetime(policy_data['end_date'], errors='coerce')

# fill missing end date with today
policy_data['end_date'] = policy_data['end_date'].fillna(pd.Timestamp.today().normalize())

# new df to add to
final = data.copy()

# create active policy flags
for idx, row in policy_data.iterrows():
    col_name = row['policy_name'].replace(' ', '_').lower() + '_active'
    final[col_name] = ((final['datetime'] >= row['effective_date']) & 
                          (final['datetime'] < row['end_date'])).astype(int)

In [4]:
# view
final.head()

,datetime,lat,lng,district,subject_age,violation,arrest_made,citation_issued,search_conducted,speed,...,texas_hb_3791_active,eo_13688_active,texas_hb_1036_active,eo_13809_active,san_antonio_hands-free_ordinance_active,texas_hb_3389_active,dhs_johnson_memo_active,fy2017_byrne_justice_assistance_grant_(jag)_active,maryland_v._wilson_active,eo_13773_active
0,2014-01-01 23:35:00,29.419450,-98.456763,4140.0,38.0,CROSSING AGAINST PEDESTRIAN CONTROL SIGNAL|PED...,0,1,0,NaN,...,0,0,0,0,0,1,0,0,1,0
1,2014-01-01 22:48:00,29.511846,-98.538947,7210.0,47.0,DISREGARD STOP SIGN|OPERATING A MOTOR VEHICLE ...,0,1,0,NaN,...,0,0,0,0,0,1,0,0,1,0
2,2014-01-01 08:30:00,29.549645,-98.463889,3130.0,27.0,OPERATING A MOTOR VEHICLE WITHOUT A VALID LICENSE,0,1,0,NaN,...,0,0,0,0,0,1,0,0,1,0
3,2014-01-01 07:51:00,29.352803,-98.529048,6150.0,27.0,SPEEDING-POSTED LIMIT|DRIVING WITHOUT PROOF OF...,0,1,0,78.0,...,0,0,0,0,0,1,0,0,1,0
4,2014-01-01 12:27:00,29.424226,-98.442586,4160.0,28.0,OPERATING A MOTOR VEHICLE WITHOUT A VALID LICE...,0,1,0,NaN,...,0,0,0,0,0,1,0,0,1,0


In [5]:
# save csv
final.to_csv('teamb_final_data.csv', index=False)